# Lesson 15: Research and Outline Agents

The **first 2 agents** of our SEO pipeline:

1. **Research Agent** — Searches the web, analyzes keywords and competitor content
2. **Outline Agent** — Creates a structured outline from the research results

The flow:

```
Topic --> [Research Agent] --> Research notes --> [Outline Agent] --> ContentOutline (JSON)
```

Research Agent uses **DuckDuckGoTools** for web search. Outline Agent uses **output_schema** to return structured JSON instead of free-form text.

> **Note**: Claude (Anthropic) supports using `tools` and `output_schema` together. As you learned in Lesson 7, this is one of the key reasons we chose Claude Sonnet for these agents.

## From Mini Pipeline to Production

In Lesson 13, you built a mini pipeline with nested schemas (`Section` inside `DetailedOutline`). The production pipeline uses the **same pattern** with more fields and more precise instructions.

The `ContentOutline` schema below uses `OutlineSection` nested inside it, same as the `Section` inside `DetailedOutline` you already practiced. If you understood Lesson 13, this will be familiar.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude
from agno.tools.duckduckgo import DuckDuckGoTools
from pydantic import BaseModel, Field

## Schema: ContentOutline

The **output schema** defines the data structure the Outline Agent returns.

This is the **same schema** used in the product (`agents/schemas.py`). When you pass `output_schema=ContentOutline` to an agent, the LLM returns JSON matching this structure.

Two classes:
- `OutlineSection` — One section in the article (corresponds to an H2)
- `ContentOutline` — The full outline: title, meta description, keywords, sections

In [ ]:
class OutlineSection(BaseModel):
    heading: str = Field(description="Section heading (H2)")
    subheadings: list[str] = Field(default_factory=list, description="Sub-section headings (H3)")
    key_points: list[str] = Field(description="Bullet points to cover")
    seo_keywords: list[str] = Field(default_factory=list, description="Target keywords for this section")


class ContentOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    meta_description: str = Field(description="Meta description, max 160 chars")
    target_keywords: list[str] = Field(description="Primary SEO keywords")
    sections: list[OutlineSection] = Field(description="Ordered list of content sections")
    tone: str = Field(default="informative", description="Writing tone")

## Research Agent

The first agent in the pipeline. Its job:

- **Search the web** using DuckDuckGo to gather information about the topic
- **Analyze keywords** — find primary and secondary keywords
- **Check competitors** — what's ranking highly, what gaps exist
- **Return research notes** as plain text (no JSON needed yet)

`DuckDuckGoTools()` is a built-in Agno toolkit that lets the agent search the web when it needs information.

In [ ]:
research_agent = Agent(
    name="Research Agent",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=[DuckDuckGoTools()],
    instructions=[
        "You are an expert SEO researcher.",
        "Research the given topic using web search.",
        "Identify primary and secondary keywords, analyze what top-ranking content covers, "
        "and find content gaps.",
        "Return your findings as clear, organized research notes.",
    ],
)

## Outline Agent

The second agent. Takes research notes from the Research Agent and creates a structured outline.

Key points:
- Uses `output_schema=ContentOutline` — the agent **must** return JSON in the defined format
- **No tools needed** — the Outline Agent only processes text, no web search or API calls
- Outline includes 5-8 sections with H2, H3, key points, and keywords per section

> **Recall**: `output_schema` is how Agno forces the LLM to return structured data. Instead of free-form text, you get a Pydantic object with `.title`, `.sections`, etc.

In [ ]:
outline_agent = Agent(
    name="Outline Agent",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=ContentOutline,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes about a topic, create a structured content outline.",
        "Include 5-8 sections with clear H2 headings, optional H3 subheadings, "
        "key points, and relevant SEO keywords per section.",
        "The title should be SEO-optimized and compelling.",
        "The meta description must be under 160 characters.",
    ],
)

## Test Run: Research --> Outline

Run both agents in sequence:

1. Research Agent searches the web for the topic
2. Outline Agent receives the research notes and creates an outline

Same pipeline pattern as Lesson 13's mini pipeline — the output of one agent is the input for the next. The real product adds database tracking and error handling on top, but the core logic is identical.

> **Cost:** ~$0.10-0.20 (2 Sonnet calls). Takes 30-60 seconds.

In [ ]:
topic = "How to optimize on-page SEO for your website"

print("Step 1: Researching...")
research = research_agent.run(f"Research this topic for an SEO article: {topic}")
print(f"Research done! ({len(research.content)} chars)\n")

print("Step 2: Creating outline...")
outline_response = outline_agent.run(
    f"Create a structured content outline from these research notes:\n\n{research.content}"
)
outline = outline_response.content

print(f"Title: {outline.title}")
print(f"Meta: {outline.meta_description}")
print(f"Keywords: {', '.join(outline.target_keywords)}")
print(f"\nSections:")
for i, section in enumerate(outline.sections, 1):
    print(f"  {i}. {section.heading}")
    for sp in section.subheadings:
        print(f"     - {sp}")

## Exercise

Change the `topic` variable to a topic relevant to your work, then re-run the test run cell above.

After the outline is generated, write code below to:
1. Print the total number of sections
2. Print the key points for the **first** section only
3. Print the meta description and check: is it under 160 characters? (use `len()`)

In [ ]:
# Exercise: Inspect the outline from the test run above

# 1. Print the total number of sections
print(f"Total sections: {___(outline.sections)}")

# 2. Print the key points for the first section only
first_section = outline.sections[___]
print(f"\nFirst section: {first_section.heading}")
print(f"Key points: {first_section.___}")

# 3. Print the meta description and check if it's under 160 characters
meta = outline.___
print(f"\nMeta description: {meta}")
print(f"Length: {___(meta)} characters")
print(f"Under 160 chars: {len(meta) ___ 160}")

<details>
<summary>Click to reveal answer</summary>

```python
# Exercise: Inspect the outline from the test run above

# 1. Print the total number of sections
print(f"Total sections: {len(outline.sections)}")

# 2. Print the key points for the first section only
first_section = outline.sections[0]
print(f"\nFirst section: {first_section.heading}")
print(f"Key points: {first_section.key_points}")

# 3. Print the meta description and check if it's under 160 characters
meta = outline.meta_description
print(f"\nMeta description: {meta}")
print(f"Length: {len(meta)} characters")
print(f"Under 160 chars: {len(meta) < 160}")
```
</details>